# Campus SVI — sample downloads

Pull a handful of actual images, to check that what the metadata calls *covered* really is
usable streetscape, or to illustrate a finding.

The rest of the project is metadata only, deliberately: coverage is measurable from positions and
dates without ever fetching a picture. **This is not a bulk collector.** Downloading a campus at
full resolution would take hours, produce gigabytes, and is needed for nothing in the analysis.
Keep the counts small.

Two ways in: **by grid cell** (a few images from one cell) or **by ID** (a specific image or
panorama you already have in mind).

---
## 1 · Setup


In [ ]:
#@title Install and load
!pip install -q geopandas pyogrio streetlevel aiohttp nest-asyncio requests pillow

import nest_asyncio; nest_asyncio.apply()   # Colab already runs a loop

import os, sys
REPO_DIR = '/content/campus-svi-acquisition'
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/aditpradana36/campus-svi-availability.git $REPO_DIR
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

from campus_svi import config, download, registry
config.set_root('/content/drive/MyDrive/campus-svi-availability')

try:
    from google.colab import userdata
    os.environ['MAPILLARY_TOKEN'] = userdata.get('MAPILLARY_TOKEN')
except Exception:
    import getpass
    os.environ['MAPILLARY_TOKEN'] = getpass.getpass('MAPILLARY_TOKEN: ')
config.MAPILLARY_TOKEN = os.environ.get('MAPILLARY_TOKEN', '')

import matplotlib.pyplot as plt
print('samples go to', config.DATA_DIR / 'samples')


---
## 2 · By grid cell

Pick a campus, see which cells actually hold records, then pull a few from one of them.


In [ ]:
#@title Which cells have records?
CAMPUS = 'ui_main'  #@param {type:'string'}
SOURCE = 'mapillary'  #@param ['mapillary', 'google']
MIN_COUNT = 3  #@param {type:'integer'}

cands = download.cells_with_coverage(CAMPUS, SOURCE, min_count=MIN_COUNT, n=12)
display(cands)


In [ ]:
#@title Download a few from one cell
GRID_ID = ''  #@param {type:'string'}
N_IMAGES = 4  #@param {type:'integer'}
GOOGLE_ZOOM = 3  #@param {type:'slider', min:0, max:5, step:1}

#@markdown `N_IMAGES = 0` means *no limit* — a dense cell can hold dozens
#@markdown of records, and each Google panorama is a stitched image. Keep
#@markdown it small unless you mean it.
#@markdown
#@markdown `GOOGLE_ZOOM` is the tile pyramid level, not a map zoom: 0 is a
#@markdown thumbnail, 5 is full resolution and slow. 3 suits a visual check.

grid_id = GRID_ID.strip() or cands['grid_id'].iloc[0]
print('cell:', grid_id)

kw = {'zoom': GOOGLE_ZOOM} if SOURCE == 'google' else {}
paths = download.sample_cell(CAMPUS, grid_id, source=SOURCE, n=N_IMAGES, **kw)


In [ ]:
#@title Look at what came back
download.contact_sheet(paths, ncols=4)
plt.show()


---
## 3 · By ID

When you already know which image or panorama you want — from a table, a map, or a previous
download — fetch it directly.

Mapillary ids are the `image_id` column of the points layer; Google ids are `pano_id`.


In [ ]:
#@title Mapillary by image id
IMAGE_IDS = ''  #@param {type:'string'}
RESOLUTION = 'thumb_1024_url'  #@param ['thumb_256_url', 'thumb_1024_url', 'thumb_2048_url', 'thumb_original_url']

#@markdown Comma-separated. `thumb_original_url` needs a token with the right
#@markdown scope; 1024 or 2048 is plenty for checking a streetscape.

ids = [i.strip() for i in IMAGE_IDS.split(',') if i.strip()]
if ids:
    out = download.download_mapillary(ids, CAMPUS, resolution=RESOLUTION)
    download.contact_sheet(out); plt.show()
else:
    print('Paste one or more image ids above.')


In [ ]:
#@title Google by pano id
PANO_IDS = ''  #@param {type:'string'}
ZOOM = 3  #@param {type:'slider', min:0, max:5, step:1}

pids = [i.strip() for i in PANO_IDS.split(',') if i.strip()]
if pids:
    out = download.download_google(pids, CAMPUS, zoom=ZOOM)
    download.contact_sheet(out); plt.show()
else:
    print('Paste one or more pano ids above.')


---
## 4 · Finding ids to try

If you have no id in mind, take a few from the points layer with their dates and contributors,
and pick from there.


In [ ]:
#@title Browse records
rows = download.pick(CAMPUS, source=SOURCE, n=15)
cols = ([c for c in ['image_id','captured_date','creator_username','camera_type','is_pano']
         if c in rows.columns] if SOURCE == 'mapillary' else
        [c for c in ['pano_id','date','capture_source','is_third_party','street_name']
         if c in rows.columns])
display(rows[cols])


---
## Notes

**Licensing.** Mapillary imagery is CC BY-SA; credit the contributor if you reproduce a frame.
Google Street View imagery is not openly licensed — use it to check your own work, and clear any
reproduction in a publication before relying on it.

**Files** land in `data/samples/{campus}/` on Drive and are not part of any pipeline output. Delete
them freely; nothing reads them.

**If a Mapillary download 401s**, the token lacks the scope for that resolution — drop to
`thumb_1024_url`. **If a Google pano fails**, it may have been removed since your metadata was
collected, which is itself worth noting if it happens often.


---
### If Google downloads fail

`asyncio.run() cannot be called from a running event loop` means the synchronous streetlevel
path was used. Colab already runs an event loop, so this module calls streetlevel's **async**
entry points instead. If you see that error, the install cell did not run — it applies
`nest_asyncio`, which is what lets the async calls share the notebook's loop.

`not found` for a pano id means the panorama has been removed since your metadata was collected.
Worth noting if it happens often: it is a real limitation of Google coverage as a data source,
not a bug in the fetch.
